# SpMV kernel version comparison

Compares the three multicore SpMV kernels on the same set of matrices:

| version | kernel | bench | csv |
|---|---|---|---|
| `multicore` | `run_spmv` | `spmv` | `bench_results_spmv_128_fp16.csv` |
| `multicube` | `run_spmv_multi_cube` | `spmv_multi_cube` | `bench_results_spmv_multi_cube_128_fp16.csv` |
| `spmv_v2` | `run_spmv_v2` | `spmv_v2` | `bench_results_spmv_v2_128_fp16.csv` |

Generate the data with:
```bash
make -f Makefile.spmv.mk profile_fp16_spmv_versions
```
By default the CSVs are written to the current directory (`TCUSCAN_BENCHMARK_REPORT_PATH`
defaults to `.`), which is what `benchmarks_dir` below points at. Set both to a
different directory if you want to keep runs separate.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
from os.path import join

benchmarks_dir = "."

# (glob pattern, version label). The patterns are mutually exclusive:
# *spmv_128* does not match spmv_multi_cube_128 nor spmv_v2_128.
version_globs = [
    ("*spmv_128*.csv", "multicore"),
    ("*spmv_multi_cube_128*.csv", "multicube"),
    ("*spmv_v2_128*.csv", "spmv_v2"),
]

dfs = []
for pattern, version in version_globs:
    files = glob(join(benchmarks_dir, pattern))
    if not files:
        print(f"WARNING: no files match {pattern}")
        continue
    df = pd.concat(map(pd.read_csv, files))
    df["version"] = version
    dfs.append(df)

grand_df = pd.concat(dfs)
grand_df = grand_df.sort_values(by=["nnz", "version"])
grand_df.head(20)

In [ ]:
# Per-matrix bar chart: time [us], one bar per version.
palette = ["#91bfdb", "#fee090", "#d73027"]

plt.figure(figsize=(14, 3))
ax = sns.barplot(
    data=grand_df,
    x="benchname",
    y="time_us",
    hue="version",
    hue_order=["multicore", "multicube", "spmv_v2"],
    edgecolor="black",
    palette=palette,
    estimator="median",
    width=0.75,
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.grid(linestyle="--", axis="y")
plt.legend(loc="upper left", ncol=3)
plt.xticks(rotation=20, ha="right")
plt.xlabel("")
plt.ylabel("time [us]")
plt.yscale("log")
plt.savefig("spmv_versions_comparison.pdf", bbox_inches="tight")

In [ ]:
# Speedup of each version relative to the multicore baseline, per matrix.
med = grand_df.groupby(["benchname", "version"])["time_us"].median().unstack()
speedup = med.div(med["multicore"], axis=0)
speedup = speedup.rename(columns=lambda c: f"{c}_x_over_multicore")
speedup.sort_values("multicube_x_over_multicore")